# 02 — Warehouse Ingestion

Loads TPC-DS CSV files from ADLS into a **Fabric Warehouse** using
`COPY INTO` (T-SQL). The Warehouse has a single configuration
(standard, no Delta-specific options).

**Prerequisites**
- CSV files uploaded to ADLS / Lakehouse Files (`Files/tpcds/sfXX/`)
- A `FabLab_Warehouse` Warehouse created via `provision/setup_fabric.py`
- This notebook run against a Fabric Spark environment with access to
  the Warehouse SQL endpoint via `%sql` magic or a `pyodbc` connection

In [ ]:
# Parameters — adjust before running
SCALE_FACTOR   = 'sf10'     # 'sf10' | 'sf100' | 'sf1000'
LAKEHOUSE_NAME = 'LH_01'
WAREHOUSE_NAME = 'WH_01'
SCHEMA         = 'benchmark'

# Base ADLS path to CSV files (served from the Lakehouse Files section)
FILES_PATH = f'abfss://{LAKEHOUSE_NAME}@onelake.dfs.fabric.microsoft.com/Files/tpcds/{SCALE_FACTOR}'

TABLES = [
    'store_sales', 'date_dim', 'item', 'store', 'customer',
    'customer_demographics', 'promotion', 'household_demographics'
]

CSV_DELIMITER = '|'

print(f'Scale factor : {SCALE_FACTOR}')
print(f'Source path  : {FILES_PATH}')
print(f'Target       : {WAREHOUSE_NAME}.{SCHEMA}')

## Step 1 — Create schema and tables

Run the cell below using the `%%sql` magic connected to the Warehouse.
Schemas and tables are created only if they do not already exist.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Helper: run T-SQL against the Warehouse via the built-in Fabric SQL connector
# Requires the notebook to be connected to the Warehouse endpoint
def run_tsql(sql: str) -> None:
    spark.sql(sql)   # Works when the notebook default lakehouse is set to the Warehouse


run_tsql(f'CREATE SCHEMA IF NOT EXISTS {SCHEMA}')
print(f'Schema {SCHEMA!r} ready.')

## Step 2 — Create tables using CTAS from CSV files

In [ ]:
# Read each CSV with Spark and infer schema, then write to the Warehouse
# using the Fabric Spark connector (spark.write with warehouse connector)

for table in TABLES:
    src_path = f'{FILES_PATH}/{table}.csv'
    target   = f'{SCHEMA}.{table}'
    print(f'Loading {target} from {src_path} ...')

    df = (
        spark.read
        .option('delimiter', CSV_DELIMITER)
        .option('header', 'false')
        .option('inferSchema', 'true')
        .csv(src_path)
    )

    (
        df.write
        .format('com.microsoft.fabric.spark.write')   # Fabric Warehouse Spark connector
        .option('workspaceId', spark.conf.get('trident.artifact.workspace.id'))
        .option('artifactId',  spark.conf.get('trident.artifact.id'))          # Warehouse item ID
        .option('schema',      SCHEMA)
        .option('table',       table)
        .mode('overwrite')
        .save()
    )
    print(f'  Done — {df.count():,} rows')

print('\nAll tables loaded into Warehouse.')

## Step 3 — Verify row counts

In [ ]:
print(f'Row counts in {WAREHOUSE_NAME}.{SCHEMA} — {SCALE_FACTOR}:')
for table in TABLES:
    count = spark.sql(f'SELECT COUNT(*) AS n FROM {SCHEMA}.{table}').collect()[0]['n']
    print(f'  {table:<35} {count:>15,}')